# 06 - Extraction de la trajectoire globale

**Objectif de l'etape.** Executer le pipeline sans annotations et extraire la trajectoire globale de la voiture.

**Methode.** La bounding box estimee est mise a jour par le mouvement moyen Lucas-Kanade. Les centres successifs de cette bbox forment la trajectoire.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

DATASET_DIR = PROJECT_ROOT / 'data' / 'car' / 'car-11' / 'img'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

image_files = sorted([p for p in DATASET_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
print('Nombre de frames:', len(image_files))

Nombre de frames: 1661


In [2]:
# Bbox manuelle de depart: ajuster si necessaire ou utiliser cv2.selectROI dans l'interface Tkinter.
# Format: x, y, w, h. Cette valeur ne vient pas d'annotations.
initial_bbox = (535, 300, 220, 105)
print('initial_bbox =', initial_bbox)

initial_bbox = (535, 300, 220, 105)


In [3]:
from src.pipeline import run_tracking_without_groundtruth

result = run_tracking_without_groundtruth(
    start_frame=0,
    end_frame=120,
    initial_bbox=initial_bbox,
    use_segmentation=True,
    preprocess_method='stretching',
    segmentation_method='otsu',
    invert_mask=True,
    save_outputs=True,
)
trajectory_df = result['trajectory_df']
display(trajectory_df.head())
print('CSV sauvegarde:', RESULTS_DIR / 'trajectory.csv')
print('Visualisation trajectoire:', RESULTS_DIR / 'final_visualization' / 'final_trajectory.png')

,frame,x,y,bbox_x,bbox_y,bbox_w,bbox_h,dx,dy,speed_px_per_frame,direction_deg,tracked_points
0,0,641.0,359.5,542,314,198,91,0.000000,0.000000,0.000000,0.000000,34
1,1,641.0,359.5,542,314,198,91,0.321840,-0.089877,0.334154,-15.602901,34
2,2,641.0,359.5,542,314,198,91,-0.321149,-0.177551,0.366962,-151.063412,30
3,3,640.0,367.0,541,329,198,76,-0.672940,-0.094154,0.679495,-172.035191,29
4,4,640.0,367.0,541,329,198,76,0.447496,-0.172917,0.479743,-21.127042,27


CSV sauvegarde: c:\Users\espacegamers\Desktop\Master IAII\Cours\drive-download-20251005T182208Z-1-001\S2\Traitement d_images et vision par ordinateur_\Projet\motion-estimation-project\results\trajectory.csv
Visualisation trajectoire: c:\Users\espacegamers\Desktop\Master IAII\Cours\drive-download-20251005T182208Z-1-001\S2\Traitement d_images et vision par ordinateur_\Projet\motion-estimation-project\results\final_visualization\final_trajectory.png


**Resultats generes.** `results/trajectory.csv`, images de trajectoire dans `results/trajectory/` et visualisation finale dans `results/final_visualization/final_trajectory.png`.

**Interpretation.** La trajectoire globale est obtenue par les centres successifs de la voiture estimee. Une trajectoire lisse indique un suivi stable; des ruptures peuvent indiquer une perte de points ou une segmentation imparfaite.